# SOC fix — stop the model over-using the batteries

**The problem.** The demand-responsive model (`itransformer_rayenfd`) meets a simulated
load increase, but it makes the **batteries** do most of the extra work. Batteries can
change output fastest, and the head splits each demand correction by how fast each
source can move — so batteries take the biggest share. They have a fixed energy tank
(4,736 MWh), so leaning on them drains it past empty/full → the state-of-charge (SOC)
constraint breaks.

**The fix (one model).** Two changes, both inside `lib/models.py::RayenHeadFixedD`:

1. **`--fd-alloc share`** — split the correction by each source's *share of the energy
   mix* (coal is biggest, so coal carries most), not by raw speed.
2. **`--soc-shield on`** — track the battery's charge level through the run and hard-cap
   its power each step so the tank can never overfill or drain.

This notebook runs **the baseline (problem)** and **`share`+`shield` (fix)** — nothing
else — then shows the verdict table and the stacked-dispatch figure.


In [ ]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""   # PRIVATE repo: paste a fine-grained READ-ONLY token (Contents: read)
BRANCH = "claude/model-bottlenecks-constraints-gb1aoj"
import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("energy_modelling"):
    !git clone -q --branch $BRANCH $url
%cd energy_modelling
!git pull -q
!nvidia-smi -L


## 1 · The problem — baseline model

`increase --g 10` raises free-window (11:00–14:00) demand by 10%; `--g 30` by 30%.
Watch `soc_worst_day_pct` cross 100 (tank overflows) and `batt_dis_resp_pct` balloon.
Each rollout is a few minutes on CPU.


In [ ]:
!python demand_simulation/study_shift.py --scenario increase --g 10 --models rayenfd
!python demand_simulation/study_shift.py --scenario increase --g 30 --models rayenfd


## 2 · The fix — `share` split + SOC `shield` (one model)


In [ ]:
!python demand_simulation/study_shift.py --scenario increase --g 10 --models rayenfd --fd-alloc share --soc-shield on
!python demand_simulation/study_shift.py --scenario increase --g 30 --models rayenfd --fd-alloc share --soc-shield on


## 3 · Verdict — before vs after

PASS = still delivers the full increase (`capture` ≈ 1) **and** the battery stays inside
its tank every day (`soc worst day` ≤ 100%).


In [ ]:
import glob, json
import pandas as pd
rows = []
for p in sorted(glob.glob("demand_simulation/sweep_eqnd/study/increase_g*_*rayenfd*.json")):
    r = json.load(open(p))
    fixed = "[share]" in r["model"] and "[soc]" in r["model"]
    rows.append({
        "model": "FIX (share+shield)" if fixed else "baseline",
        "g%": int(r["g"]),
        "capture": round(r["response_capture"], 2),
        "coal resp %": round(r["coal_resp_pct"], 1),
        "batt_dis resp %": round(r["batt_dis_resp_pct"], 1),
        "soc worst day %": round(r["soc_worst_day_pct"], 1),
        "soc days ok %": round(r["soc_day_feasible_pct"], 1),
        "closed-loop WAPE": round(r["base_WAPE"], 2),
        "gate": "PASS" if (r["soc_worst_day_pct"] <= 100 and r["soc_day_feasible_pct"] >= 95) else "FAIL",
    })
display(pd.DataFrame(rows).sort_values(["g%", "model"]).reset_index(drop=True))


## 4 · Figure — 4-day all-energy dispatch stack (the fixed model)

Three panels: actual / model at normal demand / model at +10% demand. Sources stack
bottom-up, wind+solar on top, battery charging below zero; the gold band is the
free window; the dashed line is the net demand the fleet must meet. In the fixed
model the scenario panel should look like the actual mix — coal stays up in the gold
windows instead of collapsing onto the batteries.

(Renewable layers: exact wind/solar+curtailment if `data/renewables_extract_hist.parquet`
exists — build it once locally with `python script/export_renewables_extract.py`;
otherwise a derived wind+solar band.)


In [ ]:
!python demand_simulation/study_stack_4day.py --model itransformer_rayenfd \
    --scenario increase --g 10 --fd-alloc share --soc-shield on
import glob, os
from IPython.display import Image, display
figs = sorted(glob.glob("demand_simulation/sweep_eqnd/study/figure/stacked_4day_*share*soc*.png"),
              key=os.path.getmtime)
display(Image(figs[-1]))


## What the columns mean (concrete definitions)

At each 5-min step the model outputs six numbers — the MW from
`[hydro, coal, gas_steam, gas_ocgt, battery_charging, battery_discharging]`. We run the
model **twice** over the free window: `base` (normal demand) and `scen` (raised demand),
and average each source over those steps. `mean_base(i)` / `mean_scen(i)` = source *i*'s
average MW in each run.

- **capture** = (extra net supply the fleet produced) ÷ (extra net demand we asked for),
  both in MW. `1.0` = the fleet exactly covered the added load. *This is the "did it
  respond?" metric.*
- **`<source> resp %`** = 100 × (mean_scen(i) − mean_base(i)) ÷ mean_base(i). The percent
  change in **that one source's** average output when demand rose. `batt_dis resp % = 53`
  means battery discharge ran 53% higher than its own baseline — i.e. the battery
  absorbed the extra load. *This is the "who did the work?" metric — battery over-use
  shows up here.* Note a % is relative to each source's own baseline: at g=30 the battery
  baseline is tiny, so its % looks huge (300–460%) even for a modest MW change — which is
  why the safety check is the next column (absolute energy), not this percent.
- **soc worst day %** = the largest single-day battery energy swing ÷ tank size
  (4,736 MWh). Over 100% = the tank isn't big enough = infeasible. *This is the SOC
  safety gate, in absolute energy.*
- **closed-loop WAPE** = average forecast error when the model runs on its own outputs
  (lower = better). It falls because `share` fixes the mix, not just the total.
